# 📓 Notebook 4 — Escenarios y Mecánicas de Juego

**Sección 2 — Modelación y Gamificación** · Día 4 (mañana)

### Objetivo de hoy
Implementar **escenarios de desarrollo** (ciudad verde, expansión industrial, desarrollo equilibrado...) y **mecánicas de juego interactivas**: decisiones del jugador con consecuencias, un sistema de puntuación/indicadores, y testing con retroalimentación entre equipos.

Partimos del prototipo de ayer (mapa + controles) y lo convertimos en un juego jugable de principio a fin.


## 0. Conectemos con lo anterior

Ayer construyeron un prototipo con un solo mapa inicial genérico. Hoy vamos a generar **varios mapas de partida distintos**, cada uno representando una visión de desarrollo diferente para la misma ciudad.

**✏️ Antes de ver el código, respondan:**

1. Si tuvieran que dibujar a mano un mapa de "ciudad verde" y otro de "expansión industrial" (ambos de 10×10 celdas), ¿qué porcentaje de cada tipo de celda (agua, vegetación, urbano, industrial) le pondrían a cada uno? Anoten un número aproximado para cada uno.
2. ¿Cuál de los dos escenarios creen que será más difícil de jugar bien — es decir, mantener altos los tres indicadores (calidad de vida, biodiversidad, economía) al mismo tiempo? ¿Por qué?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import ipywidgets as widgets
from IPython.display import display, clear_output
import json, os, importlib.util

np.random.seed(1)

TIPOS_CELDA = {
    0: {"nombre": "agua", "emoji": "💧", "color": "#3b82c4"},
    1: {"nombre": "vegetacion", "emoji": "🌳", "color": "#4caf6b"},
    2: {"nombre": "urbano", "emoji": "🏠", "color": "#9e9e9e"},
    3: {"nombre": "industrial", "emoji": "🏭", "color": "#e08a2c"},
}
CMAP = ListedColormap([TIPOS_CELDA[i]["color"] for i in range(len(TIPOS_CELDA))])

def dibujar_mapa(grid, titulo="Mapa"):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(grid, cmap=CMAP, vmin=0, vmax=len(TIPOS_CELDA) - 1)
    ax.set_title(titulo)
    ax.set_xticks(range(grid.shape[1])); ax.set_yticks(range(grid.shape[0]))
    ax.grid(color="white", linewidth=1.5)
    ax.tick_params(length=0, labelbottom=False, labelleft=False)
    plt.show()

# Recuperamos la clase EcosistemaUrbano del Notebook 2, si existe
if os.path.exists("simulacion_ecosistema.py"):
    spec = importlib.util.spec_from_file_location("simulacion_ecosistema", "simulacion_ecosistema.py")
    sim_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(sim_mod)
    EcosistemaUrbano = sim_mod.EcosistemaUrbano
    print("✅ EcosistemaUrbano importada desde simulacion_ecosistema.py")
else:
    class EcosistemaUrbano:
        # Copia mínima (ver Notebook 2 para la versión completa y comentada)
        def __init__(self, modelo, ruido=0.02):
            self.variables_info = {v["nombre"]: v for v in modelo["variables"]}
            self.relaciones = modelo["relaciones"]
            self.ruido = ruido
            self.estado = {v["nombre"]: float(v["valor_inicial"]) for v in modelo["variables"]}
            self.historial = {nombre: [valor] for nombre, valor in self.estado.items()}

        def _normalizar(self, nombre, valor):
            minimo, maximo = self.variables_info[nombre]["rango"]
            return 0.0 if maximo == minimo else (valor - minimo) / (maximo - minimo)

        def _clip(self, nombre, valor):
            minimo, maximo = self.variables_info[nombre]["rango"]
            return max(minimo, min(maximo, valor))

        def calcular_efectos(self):
            efectos = {nombre: 0.0 for nombre in self.estado}
            for r in self.relaciones:
                origen_norm = self._normalizar(r["origen"], self.estado[r["origen"]])
                efectos[r["destino"]] += r["signo"] * r["fuerza"] * origen_norm
            return efectos

        def paso(self, acciones=None):
            acciones = acciones or {}
            efectos = self.calcular_efectos()
            nuevo_estado = {}
            for nombre, valor in self.estado.items():
                minimo, maximo = self.variables_info[nombre]["rango"]
                escala = maximo - minimo
                cambio = efectos[nombre] * escala * 0.15 + acciones.get(nombre, 0.0) + np.random.normal(0, self.ruido) * escala
                nuevo_estado[nombre] = self._clip(nombre, valor + cambio)
            self.estado = nuevo_estado
            for nombre, valor in self.estado.items():
                self.historial[nombre].append(valor)
            return self.estado
    print("⚠️ Usando copia mínima de EcosistemaUrbano")

if os.path.exists("modelo_mental.json"):
    with open("modelo_mental.json", "r", encoding="utf-8") as f:
        modelo_base = json.load(f)
else:
    modelo_base = {
        "equipo": "Equipo de ejemplo", "zona_de_estudio": "Zona de ejemplo",
        "variables": [
            {"nombre": "vegetacion", "descripcion": "Cobertura vegetal", "unidad": "%", "rango": [0, 100], "valor_inicial": 60},
            {"nombre": "construccion", "descripcion": "Área construida", "unidad": "%", "rango": [0, 100], "valor_inicial": 30},
            {"nombre": "poblacion", "descripcion": "Habitantes (miles)", "unidad": "miles", "rango": [0, 500], "valor_inicial": 100},
            {"nombre": "contaminacion", "descripcion": "Contaminación", "unidad": "índice", "rango": [0, 100], "valor_inicial": 20},
            {"nombre": "agua", "descripcion": "Calidad de agua", "unidad": "índice", "rango": [0, 100], "valor_inicial": 70},
        ],
        "relaciones": [
            {"origen": "construccion", "destino": "vegetacion", "signo": -1, "fuerza": 0.4},
            {"origen": "construccion", "destino": "poblacion", "signo": 1, "fuerza": 0.3},
            {"origen": "poblacion", "destino": "contaminacion", "signo": 1, "fuerza": 0.3},
            {"origen": "vegetacion", "destino": "contaminacion", "signo": -1, "fuerza": 0.35},
            {"origen": "contaminacion", "destino": "agua", "signo": -1, "fuerza": 0.3},
            {"origen": "agua", "destino": "vegetacion", "signo": 1, "fuerza": 0.25},
        ],
    }

print(f"Modelo base listo — Equipo: {modelo_base['equipo']}")


## 1. Definiendo escenarios de desarrollo

Un **escenario** es un mapa inicial + una "personalidad" del sistema (qué tan rápido crecen ciertas cosas). Definimos 3 escenarios base — pueden ajustarlos o agregar los suyos:

- 🌳 **Ciudad verde**: prioriza vegetación y agua, crecimiento urbano lento.
- 🏭 **Expansión industrial**: crecimiento económico rápido, a costa de vegetación y contaminación.
- ⚖️ **Desarrollo equilibrado**: un punto medio entre ambos extremos.


In [ ]:
def generar_mapa(n, probabilidades, semilla):
    rng = np.random.default_rng(semilla)
    opciones = list(probabilidades.keys())
    pesos = list(probabilidades.values())
    return rng.choice(opciones, size=(n, n), p=pesos)

ESCENARIOS = {
    "ciudad_verde": {
        "nombre": "🌳 Ciudad verde",
        "descripcion": "Prioriza áreas verdes y cuerpos de agua; el crecimiento urbano es lento y cuidadoso.",
        "n": 10,
        "probabilidades": {0: 0.15, 1: 0.55, 2: 0.25, 3: 0.05},
        "semilla": 1,
        "factor_construccion": 0.5,   # qué tan rápido crece lo construido (menor = más lento)
    },
    "expansion_industrial": {
        "nombre": "🏭 Expansión industrial",
        "descripcion": "Crecimiento económico acelerado; la industria y la vivienda avanzan rápido sobre la vegetación.",
        "n": 10,
        "probabilidades": {0: 0.05, 1: 0.20, 2: 0.35, 3: 0.40},
        "semilla": 2,
        "factor_construccion": 1.6,
    },
    "desarrollo_equilibrado": {
        "nombre": "⚖️ Desarrollo equilibrado",
        "descripcion": "Un punto medio: crecimiento moderado con inversión pareja en vegetación, vivienda e industria.",
        "n": 10,
        "probabilidades": {0: 0.10, 1: 0.35, 2: 0.35, 3: 0.20},
        "semilla": 3,
        "factor_construccion": 1.0,
    },
}

for clave, esc in ESCENARIOS.items():
    print(f"{esc['nombre']} ({clave}): {esc['descripcion']}")
    print(f"   Probabilidades reales: {esc['probabilidades']}")


### 🔍 Observen y reflexionen

**✏️ Comparen las `probabilidades` reales de cada escenario (impresas arriba) con los porcentajes que ustedes propusieron en la Sección 0.**

- ¿Qué tan cerca estuvieron sus números de los del código? *(reemplazar)*
- La `expansion_industrial` tiene solo 5% de agua y 20% de vegetación. ¿Su predicción fue igual de extrema o más moderada? *(reemplazar)*


In [ ]:
selector_escenario = widgets.Dropdown(
    options=[(esc["nombre"], clave) for clave, esc in ESCENARIOS.items()],
    description="Escenario:",
)
display(selector_escenario)


## 2. Catálogo de acciones del jugador (con costos y efectos)

Cada acción que puede tomar el jugador tiene un **costo** (de un presupuesto limitado por turno) y un **efecto** sobre el mapa y las variables. Esto convierte la simulación libre de ayer en un juego con decisiones reales: *no se puede hacer todo, hay que elegir*.

**🤔 Antes de ver los números:** de las 4 acciones (reforestar, vivienda, fábrica, planta de tratamiento), ¿cuál creen que debería costar más y cuál menos? ¿Por qué?


In [ ]:
CATALOGO_ACCIONES = {
    "reforestar": {"nombre": "🌳 Reforestar celda", "costo": 10, "nuevo_tipo_celda": 1},
    "vivienda": {"nombre": "🏠 Construir vivienda", "costo": 15, "nuevo_tipo_celda": 2},
    "fabrica": {"nombre": "🏭 Instalar fábrica", "costo": 20, "nuevo_tipo_celda": 3},
    "planta_tratamiento": {"nombre": "🚰 Planta de tratamiento de agua", "costo": 25, "nuevo_tipo_celda": 0},
}

PRESUPUESTO_POR_TURNO = 30

for clave, accion in CATALOGO_ACCIONES.items():
    print(f"{accion['nombre']}: costo {accion['costo']} (presupuesto por turno: {PRESUPUESTO_POR_TURNO})")


## 3. Sistema de indicadores / puntuación

Calculamos 3 indicadores a partir del mapa y la simulación — el "marcador" del juego:

- **Calidad de vida**: combina vivienda suficiente, poca contaminación y agua disponible.
- **Biodiversidad**: proporción de vegetación y agua sana.
- **Economía**: proporción de uso industrial/urbano y población.

Cada uno va de 0 a 100. El reto del juego es **subir los tres a la vez**, no solo uno.


In [ ]:
def calcular_indicadores(grid, eco_estado):
    total = grid.size
    pct_vegetacion = 100 * np.sum(grid == 1) / total
    pct_urbano = 100 * np.sum(grid == 2) / total
    pct_industrial = 100 * np.sum(grid == 3) / total
    pct_agua = 100 * np.sum(grid == 0) / total

    calidad_de_vida = max(0, min(100, 0.5 * (100 - eco_estado["contaminacion"]) + 0.3 * pct_urbano + 0.2 * eco_estado["agua"]))
    biodiversidad = max(0, min(100, 0.6 * pct_vegetacion + 0.4 * pct_agua))
    economia = max(0, min(100, 0.5 * pct_industrial + 0.3 * pct_urbano + 0.2 * (eco_estado["poblacion"] / 5)))

    return {
        "calidad_de_vida": round(calidad_de_vida, 1),
        "biodiversidad": round(biodiversidad, 1),
        "economia": round(economia, 1),
    }

print("Función calcular_indicadores() lista ✅")


## 4. El juego completo: escenario + acciones + indicadores por turno

Elijan un escenario en el menú de arriba y corran esta celda para iniciar una partida de 10 turnos (representando, por ejemplo, 10 años de decisiones).

**🤔 Antes de jugar — predicción:** con el escenario que eligieron, ¿qué indicador (calidad de vida, biodiversidad o economía) creen que terminará más alto después de 10 turnos SI NO hacen ninguna acción? ¿Y si juegan intentando balancear los tres?


In [ ]:
def iniciar_partida(clave_escenario):
    esc = ESCENARIOS[clave_escenario]
    grid = generar_mapa(esc["n"], esc["probabilidades"], esc["semilla"])
    eco = EcosistemaUrbano(modelo_base, ruido=0.02)
    return {
        "escenario": clave_escenario,
        "grid": grid,
        "eco": eco,
        "presupuesto": PRESUPUESTO_POR_TURNO,
        "turno": 0,
        "max_turnos": 10,
        "historial_indicadores": [],
    }

partida = iniciar_partida(selector_escenario.value)
print(f"Partida iniciada — Escenario: {ESCENARIOS[partida['escenario']]['nombre']}")
dibujar_mapa(partida["grid"], f"Turno 0 — {ESCENARIOS[partida['escenario']]['nombre']}")
print("Indicadores iniciales:", calcular_indicadores(partida["grid"], partida["eco"].estado))


In [ ]:
slider_fila_j = widgets.IntSlider(min=0, max=partida["grid"].shape[0]-1, description="Fila")
slider_col_j = widgets.IntSlider(min=0, max=partida["grid"].shape[1]-1, description="Columna")
selector_accion_j = widgets.Dropdown(options=[(a["nombre"], c) for c, a in CATALOGO_ACCIONES.items()], description="Acción")
boton_aplicar_j = widgets.Button(description="Aplicar (gasta presupuesto)", button_style="info")
boton_turno_j = widgets.Button(description="⏭️ Terminar turno", button_style="success")
salida_j = widgets.Output()

def refrescar_juego():
    with salida_j:
        clear_output(wait=True)
        ind = calcular_indicadores(partida["grid"], partida["eco"].estado)
        dibujar_mapa(partida["grid"], f"Turno {partida['turno']}/{partida['max_turnos']} — {ESCENARIOS[partida['escenario']]['nombre']}")
        print(f"💰 Presupuesto restante este turno: {partida['presupuesto']}")
        print(f"📊 Indicadores: calidad de vida={ind['calidad_de_vida']} | biodiversidad={ind['biodiversidad']} | economía={ind['economia']}")
        if partida["turno"] >= partida["max_turnos"]:
            print("\n🏁 ¡Partida terminada! Revisen la gráfica de evolución más abajo.")

def on_aplicar_j(b):
    accion = CATALOGO_ACCIONES[selector_accion_j.value]
    if partida["presupuesto"] < accion["costo"]:
        with salida_j:
            clear_output(wait=True)
            print("⚠️ No hay presupuesto suficiente este turno para esa acción.")
        return
    fila, col = slider_fila_j.value, slider_col_j.value
    partida["grid"][fila, col] = accion["nuevo_tipo_celda"]
    partida["presupuesto"] -= accion["costo"]
    refrescar_juego()

def on_turno_j(b):
    if partida["turno"] >= partida["max_turnos"]:
        return
    total = partida["grid"].size
    variables_mapa = {
        "vegetacion": 100 * np.sum(partida["grid"] == 1) / total,
        "construccion": 100 * np.sum((partida["grid"] == 2) | (partida["grid"] == 3)) / total,
        "agua": 100 * np.sum(partida["grid"] == 0) / total,
    }
    acciones_sim = {n: (v - partida["eco"].estado[n]) * 0.3 for n, v in variables_mapa.items()}
    partida["eco"].paso(acciones=acciones_sim)
    partida["turno"] += 1
    partida["presupuesto"] = PRESUPUESTO_POR_TURNO
    partida["historial_indicadores"].append(calcular_indicadores(partida["grid"], partida["eco"].estado))
    refrescar_juego()

boton_aplicar_j.on_click(on_aplicar_j)
boton_turno_j.on_click(on_turno_j)

display(widgets.VBox([
    widgets.HBox([slider_fila_j, slider_col_j]),
    widgets.HBox([selector_accion_j, boton_aplicar_j, boton_turno_j]),
    salida_j,
]))
refrescar_juego()


In [ ]:
# Corran esta celda al terminar su partida (10 turnos) para ver cómo evolucionaron los indicadores.

if partida["historial_indicadores"]:
    turnos = range(1, len(partida["historial_indicadores"]) + 1)
    plt.figure(figsize=(8, 4.5))
    for clave in ["calidad_de_vida", "biodiversidad", "economia"]:
        plt.plot(turnos, [h[clave] for h in partida["historial_indicadores"]], marker="o", label=clave)
    plt.xlabel("Turno"); plt.ylabel("Indicador (0-100)")
    plt.title(f"Evolución de indicadores — {ESCENARIOS[partida['escenario']]['nombre']}")
    plt.legend(); plt.grid(alpha=0.3)
    plt.show()
else:
    print("Aún no hay turnos jugados. Usen el botón '⏭️ Terminar turno' arriba antes de graficar.")


### 🔍 Observen y reflexionen

**✏️ Comparen su partida real con la predicción de la Sección 4. Escriban su respuesta:**

- ¿Qué indicador terminó más alto? ¿Coincide con lo que predijeron? *(reemplazar)*
- ¿Cuál decisión (o falta de decisión) tuvo el mayor impacto en sus indicadores durante la partida? *(reemplazar)*
- Jueguen el mismo escenario una segunda vez tomando decisiones distintas. ¿Pudieron mejorar los tres indicadores a la vez, o siempre tuvieron que sacrificar uno? *(reemplazar)*


## 5. Testing y depuración entre equipos

Cuando terminen su partida de prueba, intercambien notebooks con otro equipo (o compartan pantalla) y completen esta lista juntos:

**Hoja de testing entre pares**

- [ ] ¿El mapa se actualiza correctamente después de cada acción?
- [ ] ¿El presupuesto se descuenta bien y se resetea cada turno?
- [ ] ¿Los tres indicadores se mantienen siempre entre 0 y 100 (nunca negativos ni mayores a 100)?
- [ ] ¿Los 3 escenarios se sienten distintos entre sí al jugarlos?
- [ ] ¿Encontraron algún bug? Descríbanlo aquí: `______________________`
- [ ] ¿Qué mecánica les gustaría agregar o cambiar? `______________________`

**Rúbrica rápida de mecánicas de juego** (califiquen de 1 a 5):

| Criterio | 1-5 |
|---|---|
| Claridad de las reglas (¿se entiende qué hacer?) | |
| Balance (¿ningún indicador domina fácilmente?) | |
| Feedback visual (¿se ve claro el efecto de cada acción?) | |
| Diversión / interés en seguir jugando | |

**🔍 Reflexionen antes de empezar el testing:** ¿por qué creen que es más útil que OTRO equipo pruebe su juego, en vez de solo probarlo ustedes mismos? Piensen en algo que ustedes ya no "ven" de su propio juego por estar tan metidos en construirlo.


In [ ]:
# Prueba automática rápida: verifica que los indicadores nunca salgan de rango.
# Corre esta celda para una auto-revisión antes del testing entre pares.

def prueba_rango_indicadores(n_turnos_prueba=15):
    p = iniciar_partida(selector_escenario.value)
    errores = []
    for t in range(n_turnos_prueba):
        p["eco"].paso()
        ind = calcular_indicadores(p["grid"], p["eco"].estado)
        for nombre, valor in ind.items():
            if not (0 <= valor <= 100):
                errores.append((t, nombre, valor))
    if errores:
        print(f"❌ Se encontraron {len(errores)} valores fuera de rango:", errores[:5])
    else:
        print(f"✅ Los indicadores se mantuvieron entre 0 y 100 durante {n_turnos_prueba} turnos simulados.")

prueba_rango_indicadores()


## 🎨 Reto: creen su propio escenario

Agreguen un cuarto escenario a `ESCENARIOS` (por ejemplo, "reconstrucción post-desastre" o "boom turístico") con su propia combinación de probabilidades y `factor_construccion`. Vuelvan a correr el juego con su escenario nuevo.


## 🧭 Bitácora de aprendizaje

**✏️ Completen cada punto como equipo:**

1. En la vida real, las ciudades no pueden "deshacer" una fábrica con un botón como en el juego. ¿Qué decisiones de su juego representan cambios fáciles de revertir en la realidad, y cuáles representan cambios casi permanentes? *(reemplazar)*
2. ¿Existe en la realidad algún equivalente al "presupuesto por turno" que limita las decisiones de una ciudad real? Den un ejemplo. *(reemplazar)*
3. Si un alcalde o alcaldesa real jugara su juego, ¿qué escenario le recomendarían para su ciudad y por qué? *(reemplazar)*
4. ¿Qué feedback del otro equipo (testing entre pares) fue el más útil y qué cambiaron gracias a él? *(reemplazar)*

## ✅ Producto esperado de esta sesión

- [ ] Al menos 3 escenarios de desarrollo jugables (`ciudad_verde`, `expansion_industrial`, `desarrollo_equilibrado`)
- [ ] Sistema de indicadores (calidad de vida, biodiversidad, economía) funcionando y visible en pantalla
- [ ] Catálogo de acciones con costos y efectos, y presupuesto por turno
- [ ] Partida completa jugada (10 turnos) con gráfica de evolución de indicadores
- [ ] Hoja de testing entre pares completada con otro equipo
- [ ] Bitácora de aprendizaje completa

**Esta tarde:** Sección 3 — integramos datos satelitales reales de su zona como condición inicial del juego. 🛰️
